In [0]:
# Step 11.2: Read Silver Delta data

silver_path = (
    "abfss://silver@stretaildataeng26.dfs.core.windows.net/"
    "retail_transactions"
)

df_gold_source = (
    spark.read
    .format("delta")
    .load(silver_path)
)

print("Total Silver records:", df_gold_source.count())
print("Total columns:", len(df_gold_source.columns))

display(df_gold_source.limit(5))

In [0]:
from pyspark.sql.functions import sum as spark_sum, round
df_total_revenue =  df_gold_source.agg(
    round(
        spark_sum("LineTotal"),
        2
    ).alias("TotalNetRevenue")
)

display(df_total_revenue)

In [0]:
from pyspark.sql.functions import (
    date_format,
    sum as spark_sum,
    round,
    col
)

# Calculate monthly net revenue
df_monthly_sales = (
    df_gold_source
    .withColumn(
        "SalesMonth",
        date_format(col("InvoiceDate"), "yyyy-MM")
    )
    .groupBy("SalesMonth")
    .agg(
        round(
            spark_sum("LineTotal"), 2
        ).alias("MonthlyRevenue")
    )
    .orderBy("SalesMonth")
)

display(df_monthly_sales)

In [0]:
from pyspark.sql.functions import col

# Find the month with the highest revenue
df_top_month = (
    df_monthly_sales
    .orderBy(col("MonthlyRevenue").desc())
    .limit(10)
)

display(df_top_month)

In [0]:
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    round
)

df_top_products = (
    df_gold_source
    .filter(col("Description").isNotNull())
    .groupBy("StockCode", "Description")
    .agg(
        round(
            spark_sum("LineTotal"), 2
        ).alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

display(df_top_products.limit(10))

In [0]:
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    round
)

# Calculate net revenue by country
df_country_sales = (
    df_gold_source
    .groupBy("Country")
    .agg(
        round(
            spark_sum("LineTotal"), 2
        ).alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

# Display the top 10 countries
display(df_country_sales.limit(10))

In [0]:
# Gold storage path
gold_revenue_path = (
    "abfss://gold@stretaildataeng26.dfs.core.windows.net/"
    "gold_total_revenue"
)

# Save total revenue in Delta format
(
    df_total_revenue.write
    .format("delta")
    .mode("overwrite")
    .save(gold_revenue_path)
)

print("Gold total revenue saved successfully!")

In [0]:
# Gold monthly sales storage path
gold_monthly_path = (
    "abfss://gold@stretaildataeng26.dfs.core.windows.net/"
    "gold_monthly_sales"
)

# Save monthly sales in Delta format
(
    df_monthly_sales.write
    .format("delta")
    .mode("overwrite")
    .save(gold_monthly_path)
)

print("Gold monthly sales saved successfully!")

In [0]:
# Gold top products storage path
gold_products_path = (
    "abfss://gold@stretaildataeng26.dfs.core.windows.net/"
    "gold_top_products"
)

# Save product revenue analysis in Delta format
(
    df_top_products.write
    .format("delta")
    .mode("overwrite")
    .save(gold_products_path)
)

print("Gold top products saved successfully!")

In [0]:
# Gold country sales storage path
gold_country_path = (
    "abfss://gold@stretaildataeng26.dfs.core.windows.net/"
    "gold_country_sales"
)

# Save country-wise revenue in Delta format
(
    df_country_sales.write
    .format("delta")
    .mode("overwrite")
    .save(gold_country_path)
)

print("Gold country sales saved successfully!")

In [0]:
gold_base = (
    "abfss://gold@stretaildataeng26.dfs.core.windows.net/"
)

gold_datasets = [
    "gold_total_revenue",
    "gold_monthly_sales",
    "gold_top_products",
    "gold_country_sales"
]

for dataset in gold_datasets:
    df = spark.read.format("delta").load(
        gold_base + dataset
    )
    print(f"{dataset}: {df.count()} records")